# prep: JD 전처리 v2 — 텍스트 정제 + URL 스크래핑 + Gemini 구조화 추출

- **목적:** `company_jobdescription_preprocessing.csv` (113,231행, 1차 필터 통과본) → user_data와 매칭에 fit한 enriched JD CSV 생성.
- **이유:** 현재 `detail_text_clean`에 모든 정보가 한 줄로 뭉쳐있고 인코딩 깨진 행 다수. 5관점 매칭 가중치를 위해 *직무·산업·스킬·역량·근무지·posting_type* 컬럼 분리 필요.
- **입력:** `/raw/data/company_jobdescription_preprocessing.csv`
- **출력:** `/raw/data/company_jobdescription_enriched.csv`
- **캐시:** `/raw/data/gemini_cache/jd_profile_v2/{job_id}.json` (Gemini 재호출 방지)
- **스크랩 캐시:** `/raw/data/jd_scraped/{job_id}.txt` (선택)
- **작성일:** 2026-05-28

## 파이프라인 (4단계)

```
preprocessing.csv (113,231행)
        │
        ▼
[1] 텍스트 정제 — ftfy 인코딩 복구 + 이모지 제거 + HTML entity decode + 비한글 노이즈 제거 + whitespace 정규화
        │
        ▼
[2] URL 추출 & 스크래핑 (SCRAPE_WEB=True 시) — detail_text_clean의 https?://... 추출 → requests+BeautifulSoup으로 본문 텍스트 보강
        │
        ▼
[3] 중복 제거 — (company_name + title + first 200 chars detail) 해시 기준 drop_duplicates
        │
        ▼
[4] Gemini 구조화 추출 — JD_PROFILE_SCHEMA로 jd_job_role / jd_industry / jd_required_skills / jd_competencies / jd_location / jd_posting_kind / jd_core_duties 등 추출
        │
        ▼
enriched.csv (원본 8컬럼 + 정제 컬럼 + Gemini 추출 컬럼)
```

## 파라미터 (cell 1에서 조정)

- `SAMPLE_SIZE`: `None`=전체, 정수=상위 N행만 (dry-run 권장: 10 → 100 → 1000 → 전체)
- `SCRAPE_WEB`: `True`=URL 따라가서 스크랩, `False`=URL만 저장하고 스크랩 skip
- `GEMINI_MODEL`: 기본 `gemini-2.5-flash` (속도/비용 균형)
- `MAX_WORKERS`: Gemini 호출 동시성 (rate-limit 고려, 기본 1)

In [33]:
# 필요 시 의존성 설치 (주석 해제)
# !pip install -q ftfy emoji beautifulsoup4 lxml google-genai tqdm requests python-dotenv

import os, re, json, time, html, hashlib
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

# 선택 의존성 (없으면 fallback)
try:
    import ftfy
    HAS_FTFY = True
except ImportError:
    HAS_FTFY = False
    print('⚠️ ftfy 미설치 — 인코딩 복구 약화. pip install ftfy 권장')

try:
    import emoji as _emoji
    HAS_EMOJI = True
except ImportError:
    HAS_EMOJI = False

try:
    import requests
    from bs4 import BeautifulSoup
    HAS_SCRAPE = True
except ImportError:
    HAS_SCRAPE = False
    print('⚠️ requests/bs4 미설치 — 스크래핑 skip. pip install requests beautifulsoup4 lxml 권장')

try:
    from google import genai
    from google.genai import types
    HAS_GEMINI = True
except ImportError:
    HAS_GEMINI = False
    print('⚠️ google-genai 미설치 — Gemini 호출 skip. pip install google-genai 필요')

try:
    from dotenv import load_dotenv
    HAS_DOTENV = True
except ImportError:
    HAS_DOTENV = False
    print('⚠️ python-dotenv 미설치 — .env 자동 로드 안 됨. pip install python-dotenv 권장 (또는 셸에서 export)')

# === .env 자동 로드 (exp-005 패턴) ===
# Jupyter cwd가 어디든 .env 찾기: cwd → cwd.parent → 절대경로 fallback
_dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('output/.env'),
]
_dotenv_loaded = False
if HAS_DOTENV:
    for _p in _dotenv_candidates:
        if _p.exists():
            load_dotenv(_p, override=False)   # 이미 export된 env 우선
            print(f'[.env loaded] {_p}')
            _dotenv_loaded = True
            break
    if not _dotenv_loaded:
        print('[.env not found] 탐색한 경로:')
        for _p in _dotenv_candidates:
            print(f'  - {_p} (exists={_p.exists()})')
else:
    print('[.env skip] python-dotenv 미설치')

# 경로
RAW_DATA = Path('raw/data')
SRC = RAW_DATA / 'company_jobdescription_preprocessing.csv'
DST = RAW_DATA / 'company_jobdescription_enriched.csv'
GEMINI_CACHE = RAW_DATA / 'gemini_cache' / 'jd_profile_v2'
SCRAPE_CACHE = RAW_DATA / 'jd_scraped'
GEMINI_CACHE.mkdir(parents=True, exist_ok=True)
SCRAPE_CACHE.mkdir(parents=True, exist_ok=True)

# ★ 파라미터 (조정 가능)
SAMPLE_SIZE = None         # ★ 전체 113,231행 처리. 캐시 hit은 즉시 통과, 새 행만 API 호출. 중단해도 이어가기 OK
SCRAPE_WEB = False         # True=URL 따라 스크랩 (느리고 실패 가능), False=URL만 저장
ENRICH_VIA_SEARCH = False  # True=회사명으로 DDG 검색해 회사 페이지 본문 보강 (mojibake/짧은 detail 행에 유효)
ENRICH_TRIGGER_LEN = 150   # detail_text_clean_v2 길이가 이 미만이면 enrichment 시도
GEMINI_MODEL = 'gemini-2.5-flash'
MAX_WORKERS = 1            # rate-limit 위험 시 1 유지
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '') or os.environ.get('GOOGLE_API_KEY', '')

assert SRC.exists(), f'원본 없음: {SRC}'
if not GEMINI_API_KEY:
    print('⚠️ GEMINI_API_KEY/GOOGLE_API_KEY 환경변수 비어있음 — Gemini 단계는 캐시된 행만 처리됨')

# 마스킹 출력 (앞 4자 + ...)
_key_mask = (GEMINI_API_KEY[:4] + '...' + GEMINI_API_KEY[-2:]) if GEMINI_API_KEY else '(미설정)'
print(f'SRC               : {SRC}')
print(f'DST               : {DST}')
print(f'SAMPLE_SIZE       : {SAMPLE_SIZE}')
print(f'SCRAPE_WEB        : {SCRAPE_WEB}')
print(f'ENRICH_VIA_SEARCH : {ENRICH_VIA_SEARCH}')
print(f'GEMINI_MODEL      : {GEMINI_MODEL}')
print(f'GEMINI_API_KEY    : {_key_mask}  (HAS_GEMINI={HAS_GEMINI})')

⚠️ ftfy 미설치 — 인코딩 복구 약화. pip install ftfy 권장
⚠️ requests/bs4 미설치 — 스크래핑 skip. pip install requests beautifulsoup4 lxml 권장
[.env loaded] output/.env
SRC               : raw/data/company_jobdescription_preprocessing.csv
DST               : raw/data/company_jobdescription_enriched.csv
SAMPLE_SIZE       : None
SCRAPE_WEB        : False
ENRICH_VIA_SEARCH : False
GEMINI_MODEL      : gemini-2.5-flash
GEMINI_API_KEY    : AIza...xU  (HAS_GEMINI=True)


In [34]:
# === 텍스트 정제 함수 ===
# 핵심 원칙:
#   ✅ 보존: 회사 정체성/직무 본질/주요 업무/인재상/요구 역량/근무 환경(문화 신호)
#   ❌ 제거: 이메일/전화/접수기간/접수방법/전형단계(Step)/유의사항 boilerplate/모집인원

EMOJI_PATTERN = re.compile(
    '[\U0001F300-\U0001FAFF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF'
    '\U0001F900-\U0001F9FF\U00002600-\U000027BF\U0001FA70-\U0001FAFF]+',
    flags=re.UNICODE,
)
URL_PATTERN = re.compile(r'https?://[^\s\)\]\,\>\"\\\']+', re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b')
PHONE_PATTERN = re.compile(r'(?:\b0\d{1,2}[-.\s]?\d{3,4}[-.\s]?\d{4}\b)|(?:1[5-9]\d{2}[-.\s]?\d{4})')
WHITESPACE_PATTERN = re.compile(r'\s+')
MOJIBAKE_PATTERN = re.compile(r'[\u0080-\u00FF]{3,}')
KEEP_PATTERN = re.compile(
    r'[\uAC00-\uD7A3\u3131-\u318EA-Za-z0-9\s'
    r'\.,!?:;\-_\(\)\[\]\{\}\/\&@#%\*\+\=\'\"`~\$\^<>|]'
)

# ★ 행정/절차 noise — 의미 매칭에 무관 (사용자 결정 2026-05-28)
ADMIN_NOISE_PATTERNS = [
    # 전형 절차 / Step 패턴
    re.compile(r'전형\s*절차[\s\S]{0,400}?(?=주요업무|자격요건|우대|근무|복리|혜택|문의|$)', re.IGNORECASE),
    re.compile(r'Step\s*\d+[^\n]*'),
    re.compile(r'(?:서류전형|1차\s*면접|2차\s*면접|임원\s*면접|최종\s*합격|실무\s*면접)(?:\s*[>→·\-]\s*(?:서류전형|1차\s*면접|2차\s*면접|임원\s*면접|최종\s*합격|실무\s*면접))+'),
    # 접수 기간 / 방법
    re.compile(r'(?:지원\s*기간\s*(?:및|/)\s*방법|접수\s*기간|모집\s*기간)[\s\S]{0,200}?(?:채용\s*시\s*마감|상시\s*채용|\d{4}[\-./]\d{1,2}[\-./]\d{1,2})', re.IGNORECASE),
    re.compile(r'접수\s*방법[\s\S]{0,100}?(?:지원|간편지원|이메일|홈페이지|온라인)'),
    re.compile(r'\d{4}[\-./]\d{1,2}[\-./]\d{1,2}\s*[~\-]\s*(?:\d{4}[\-./]\d{1,2}[\-./]\d{1,2}|채용\s*시\s*마감|상시)'),
    # 유의사항 boilerplate
    re.compile(r'유의\s*사항[\s\S]{0,300}?(?:취소될\s*수\s*있습니다|불이익|기재한다)\s*[.。]?'),
    re.compile(r'허위\s*사실[\s\S]{0,80}?(?:취소|불이익)[^\n]*'),
    # 문의 이메일/연락처 라벨
    re.compile(r'문의\s*(?:이메일|연락처|전화|담당자)[\s\S]{0,80}'),
    # 모집인원 (0명) 류
    re.compile(r'\(\s*\d{1,2}\s*명\s*\)'),
    re.compile(r'모집\s*인원\s*:?\s*\d{0,3}\s*명'),
    # 회사 주소 상세 (건물명/층호)
    re.compile(r'근무지역\s*[:：]?\s*[^\n]{0,50}?(?:[가-힣]+[시군구]\s*[가-힣]+[로길동가]\s*[\d\-]+)\s*\d*\s*호?', re.IGNORECASE),
]


def is_empty(v) -> bool:
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == '' or s == '[]' or s == '[ ]':
        return True
    if s.lower() in {'nan', 'null', 'none'}:
        return True
    return False


def korean_ratio(s: str) -> float:
    if not s:
        return 0.0
    korean = sum(1 for c in s if '\uAC00' <= c <= '\uD7A3')
    return korean / max(1, len(s))


def extract_urls(text):
    if not text or pd.isna(text):
        return []
    urls = URL_PATTERN.findall(str(text))
    return [u.rstrip('.,;:)') for u in urls]


def strip_admin_noise(s: str) -> str:
    """이메일/전화/전형단계/접수기간/유의사항 등 매칭에 무관한 행정 noise만 선택 제거.
    의미 있는 본문(회사·직무·업무·인재상·역량)은 절대 건드리지 않음."""
    for pat in ADMIN_NOISE_PATTERNS:
        s = pat.sub(' ', s)
    s = EMAIL_PATTERN.sub(' ', s)
    s = PHONE_PATTERN.sub(' ', s)
    return s


def clean_text(text):
    """인코딩 복구 → HTML entity decode → 이모지 제거 → URL/이메일/전화 제거
    → 행정 noise(전형단계·접수기간·유의사항) 제거 → mojibake 제거 → whitespace.
    의미 있는 본문은 보존."""
    if is_empty(text):
        return ''
    s = str(text)
    if HAS_FTFY:
        s = ftfy.fix_text(s)
    s = html.unescape(s)
    if HAS_EMOJI:
        s = _emoji.replace_emoji(s, replace='')
    else:
        s = EMOJI_PATTERN.sub('', s)
    s = URL_PATTERN.sub(' ', s)
    s = strip_admin_noise(s)
    s = MOJIBAKE_PATTERN.sub(' ', s)
    s = ''.join(c if KEEP_PATTERN.match(c) else ' ' for c in s)
    s = WHITESPACE_PATTERN.sub(' ', s).strip()
    return s


# 자가검증: 퓨어루 JD 같은 텍스트에서 보존/제거가 의도대로 되는지
PUREU_SAMPLE = (
    '[퓨어루] 영유아 스킨케어 브랜드 운영 및 MD 주니어 모집 '
    '접수 기간 2026-04-01 ~ 채용 시 마감 '
    '담당업무 안녕하세요. 송지랩스는 무자극 영유아 스킨케어 브랜드 퓨어루를 운영하는 회사입니다. '
    '주요업무 판매 채널 운영 및 관리 자사몰 월별 프로모션 기획 '
    '자격요건 관련 분야에서 2~5년의 경력을 보유하신 분 '
    '근무지역 서울 구로구 가마산로 242 505호 '
    '전형 절차 Step 1 서류전형 Step 2 1차 면접 Step 2 최종합격 '
    '지원 기간 및 방법 접수기간 2026-04-01 ~ 채용 시 마감 접수방법 링커리어 간편지원 '
    '문의 이메일 jhpark@sj-labs.kr '
    '유의사항 허위 사실이 발견될 경우 채용이 취소될 수 있습니다.'
)
out = clean_text(PUREU_SAMPLE)
print('=== 자가검증 — 퓨어루 JD 발췌 ===')
print(f'IN  ({len(PUREU_SAMPLE)}자): {PUREU_SAMPLE[:300]}...')
print(f'\nOUT ({len(out)}자):')
print(out)
print(f'\n한글비율: {korean_ratio(out):.1%}')
removed_signals = ['Step 1', 'Step 2', '접수방법', 'jhpark@sj-labs', '유의사항', '허위 사실']
for sig in removed_signals:
    assert sig not in out, f'❌ 제거 실패: {sig!r}'
kept_signals = ['송지랩스', '영유아 스킨케어', '퓨어루', '판매 채널', '자사몰', '경력']
for sig in kept_signals:
    assert sig in out, f'❌ 보존 실패: {sig!r}'
print('\n✅ 제거(전형/접수/이메일/유의사항)·보존(회사·직무·업무) 모두 통과')

=== 자가검증 — 퓨어루 JD 발췌 ===
IN  (357자): [퓨어루] 영유아 스킨케어 브랜드 운영 및 MD 주니어 모집 접수 기간 2026-04-01 ~ 채용 시 마감 담당업무 안녕하세요. 송지랩스는 무자극 영유아 스킨케어 브랜드 퓨어루를 운영하는 회사입니다. 주요업무 판매 채널 운영 및 관리 자사몰 월별 프로모션 기획 자격요건 관련 분야에서 2~5년의 경력을 보유하신 분 근무지역 서울 구로구 가마산로 242 505호 전형 절차 Step 1 서류전형 Step 2 1차 면접 Step 2 최종합격 지원 기간 및 방법 접수기간 2026-04-01 ~ 채용 시 마감 접수방법 링커리어 간편지원 문의...

OUT (159자):
[퓨어루] 영유아 스킨케어 브랜드 운영 및 MD 주니어 모집 ~ 채용 시 마감 담당업무 안녕하세요. 송지랩스는 무자극 영유아 스킨케어 브랜드 퓨어루를 운영하는 회사입니다. 주요업무 판매 채널 운영 및 관리 자사몰 월별 프로모션 기획 자격요건 관련 분야에서 2~5년의 경력을 보유하신 분

한글비율: 69.2%

✅ 제거(전형/접수/이메일/유의사항)·보존(회사·직무·업무) 모두 통과


In [35]:
# === [1단계] 로드 + 텍스트 정제 ===
df = pd.read_csv(SRC, encoding='utf-8-sig', low_memory=False)
print(f'원본: {len(df):,} rows × {len(df.columns)} cols')
print(f'컬럼: {list(df.columns)}')

# 정제 대상 텍스트 컬럼
TEXT_COLS = ['title', 'duties_clean', 'skills_clean', 'benefits_clean', 'detail_text_clean']

for col in TEXT_COLS:
    if col not in df.columns:
        print(f'  skip (컬럼 없음): {col}')
        continue
    new_col = col + '_v2'
    df[new_col] = df[col].apply(clean_text)
    before = df[col].fillna('').astype(str).str.len().mean()
    after = df[new_col].str.len().mean()
    kr = df[new_col].apply(korean_ratio).mean()
    print(f'  {col:>22s}: avg_len {before:>6.0f} → {after:>6.0f} | 한글비율 {kr:.1%}')

# URL 추출 (detail_text_clean 원본 기준 — 정제 전에 보존)
df['extracted_urls'] = df['detail_text_clean'].apply(extract_urls)
df['has_url'] = df['extracted_urls'].apply(lambda x: len(x) > 0)
print(f'\nURL 포함 행: {df["has_url"].sum():,} / {len(df):,} ({df["has_url"].mean()*100:.1f}%)')

원본: 86,200 rows × 8 cols
컬럼: ['job_id', 'company_name', 'title', 'job_types', 'duties_clean', 'skills_clean', 'benefits_clean', 'detail_text_clean']
                   title: avg_len     29 →     29 | 한글비율 58.4%
            duties_clean: avg_len    144 →    143 | 한글비율 20.6%
            skills_clean: avg_len      0 →      0 | 한글비율 0.0%
          benefits_clean: avg_len      0 →      0 | 한글비율 0.0%
       detail_text_clean: avg_len    501 →    419 | 한글비율 53.3%

URL 포함 행: 6,413 / 86,200 (7.4%)


In [36]:
# === [2단계] URL 스크래핑 (선택) ===

SCRAPE_TIMEOUT = 10
SCRAPE_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'
}


def scrape_url(url: str, job_id: int) -> str:
    """URL 본문 텍스트 추출. 캐시 사용. 실패 시 빈 문자열."""
    cache_path = SCRAPE_CACHE / f'{job_id}.txt'
    if cache_path.exists():
        return cache_path.read_text(encoding='utf-8')

    if not HAS_SCRAPE:
        return ''
    try:
        resp = requests.get(url, headers=SCRAPE_HEADERS, timeout=SCRAPE_TIMEOUT, allow_redirects=True)
        if resp.status_code != 200:
            return ''
        # bytes → text (charset 자동 감지)
        resp.encoding = resp.apparent_encoding or 'utf-8'
        soup = BeautifulSoup(resp.text, 'lxml')
        # script/style/nav/footer 제거
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        cleaned = clean_text(text)[:4000]   # 길이 cap
        cache_path.write_text(cleaned, encoding='utf-8')
        return cleaned
    except Exception as e:
        return ''


if SCRAPE_WEB:
    target = df[df['has_url']].head(SAMPLE_SIZE) if SAMPLE_SIZE else df[df['has_url']]
    print(f'스크래핑 대상: {len(target):,} 행')
    scraped = []
    for _, row in tqdm(target.iterrows(), total=len(target), desc='scraping'):
        urls = row['extracted_urls'][:1]   # 첫 URL만
        text = scrape_url(urls[0], int(row['job_id'])) if urls else ''
        scraped.append(text)
    df.loc[target.index, 'scraped_text'] = scraped
    df['scraped_text'] = df['scraped_text'].fillna('')
    non_empty = (df['scraped_text'].str.len() > 50).sum()
    print(f'스크랩 성공: {non_empty:,} / {len(target):,}')
else:
    df['scraped_text'] = ''
    print('SCRAPE_WEB=False — 스크래핑 skip (URL만 extracted_urls 컬럼에 저장)')

SCRAPE_WEB=False — 스크래핑 skip (URL만 extracted_urls 컬럼에 저장)


In [37]:
# === [3단계] 중복 제거 ===

def dup_key(row) -> str:
    parts = [
        str(row.get('company_name', '')).strip().lower(),
        str(row.get('title_v2', '')).strip().lower()[:80],
        str(row.get('detail_text_clean_v2', '')).strip().lower()[:200],
    ]
    raw = '|'.join(parts)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]

df['_dup_hash'] = df.apply(dup_key, axis=1)
n_before = len(df)
df = df.drop_duplicates(subset=['_dup_hash'], keep='first').reset_index(drop=True)
n_after = len(df)
df = df.drop(columns=['_dup_hash'])
print(f'중복 제거: {n_before:,} → {n_after:,} ({n_before - n_after:,} 제거, {(n_before-n_after)/n_before*100:.2f}%)')

중복 제거: 86,200 → 84,930 (1,270 제거, 1.47%)


In [38]:
# === [3.5단계] 회사명 기반 웹 검색 enrichment (선택) ===
# 동작 조건: ENRICH_VIA_SEARCH=True AND (detail_text_clean_v2 < ENRICH_TRIGGER_LEN OR 한글비율 < 0.3)
# 방법: DuckDuckGo HTML 검색(회사명+title 키워드) → 상위 3 결과 URL 중 회사 공식·채용 도메인 추정
#       → 그 페이지 GET → BeautifulSoup으로 본문 텍스트 추출 → clean_text → 'company_web_text'
# 캐시: raw/data/company_web_cache/{job_id}.txt

WEB_CACHE = RAW_DATA / 'company_web_cache'
WEB_CACHE.mkdir(parents=True, exist_ok=True)

DDG_HTML_ENDPOINT = 'https://html.duckduckgo.com/html/'
DDG_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
}
# 회사 공식·채용 사이트 도메인 우선순위 (이게 결과에 있으면 우선 선택)
PREFERRED_DOMAINS = ['.co.kr', '.kr', '.com', 'wikipedia.org', 'saramin.co.kr', 'jobkorea.co.kr', 'wanted.co.kr', 'jumpit.co.kr', 'rallit.com', 'linkedin.com']
BLOCKED_DOMAINS = ['youtube.com', 'tiktok.com', 'instagram.com', 'facebook.com', 'twitter.com', 'reddit.com', 'pinterest.com']


def ddg_search(query: str, max_results: int = 5):
    """DDG HTML 검색 → [(title, url)] list. 실패 시 빈 list."""
    if not HAS_SCRAPE:
        return []
    try:
        resp = requests.post(DDG_HTML_ENDPOINT, data={'q': query}, headers=DDG_HEADERS, timeout=15)
        if resp.status_code != 200:
            return []
        soup = BeautifulSoup(resp.text, 'lxml')
        results = []
        for a in soup.select('a.result__a, a.result__url')[:max_results * 2]:
            href = a.get('href', '')
            title = a.get_text(strip=True)
            if href and 'http' in href:
                # DDG는 'uddg' 파라미터로 실제 URL 감싸기도 함
                if 'uddg=' in href:
                    from urllib.parse import urlparse, parse_qs, unquote
                    qs = parse_qs(urlparse(href).query)
                    href = unquote(qs.get('uddg', [href])[0])
                if any(b in href for b in BLOCKED_DOMAINS):
                    continue
                results.append((title, href))
                if len(results) >= max_results:
                    break
        return results
    except Exception:
        return []


def pick_best_url(results, company_name: str) -> str | None:
    """검색 결과 중 회사 공식·채용 도메인 우선 선택."""
    if not results:
        return None
    # 회사명이 도메인에 직접 포함된 결과 1순위
    company_slug = re.sub(r'[^a-zA-Z0-9가-힣]', '', company_name.lower())[:8]
    for _, url in results:
        if company_slug and company_slug in url.lower():
            return url
    # PREFERRED_DOMAINS 순서대로
    for pref in PREFERRED_DOMAINS:
        for _, url in results:
            if pref in url.lower():
                return url
    # fallback: 첫 결과
    return results[0][1]


def scrape_company_web(company_name: str, title: str, job_id: int) -> str:
    """회사명 + title로 웹 검색 → 회사 페이지 스크랩 → 정제 본문.
    캐시 사용. 실패 시 빈 문자열."""
    cache_path = WEB_CACHE / f'{job_id}.txt'
    if cache_path.exists():
        return cache_path.read_text(encoding='utf-8')

    if not HAS_SCRAPE:
        return ''

    # 검색 쿼리: 회사명 + (title의 핵심 키워드 일부)
    query = f'{company_name} 채용 회사 소개'
    results = ddg_search(query, max_results=5)
    url = pick_best_url(results, company_name)
    if not url:
        cache_path.write_text('', encoding='utf-8')
        return ''

    try:
        resp = requests.get(url, headers=DDG_HEADERS, timeout=10, allow_redirects=True)
        if resp.status_code != 200:
            cache_path.write_text('', encoding='utf-8')
            return ''
        resp.encoding = resp.apparent_encoding or 'utf-8'
        soup = BeautifulSoup(resp.text, 'lxml')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        cleaned = clean_text(text)[:3000]   # 본문 너무 길면 cap
        cache_path.write_text(cleaned, encoding='utf-8')
        time.sleep(1.0)   # rate-limit 친화
        return cleaned
    except Exception:
        cache_path.write_text('', encoding='utf-8')
        return ''


df['company_web_text'] = ''

if ENRICH_VIA_SEARCH:
    # 조건: detail 정제본이 ENRICH_TRIGGER_LEN 미만 OR 한글비율 < 30%
    df['_kr_ratio'] = df['detail_text_clean_v2'].fillna('').apply(korean_ratio)
    need_mask = (df['detail_text_clean_v2'].fillna('').str.len() < ENRICH_TRIGGER_LEN) | (df['_kr_ratio'] < 0.30)
    target = df[need_mask].head(SAMPLE_SIZE) if SAMPLE_SIZE else df[need_mask]
    print(f'enrichment 대상 (detail<{ENRICH_TRIGGER_LEN}자 OR 한글<30%): {len(target):,}/{len(df):,} 행')

    enriched_texts = []
    for _, row in tqdm(target.iterrows(), total=len(target), desc='web enrich'):
        text = scrape_company_web(
            company_name=str(row.get('company_name', '')),
            title=str(row.get('title', '')),
            job_id=int(row['job_id']),
        )
        enriched_texts.append(text)
    df.loc[target.index, 'company_web_text'] = enriched_texts
    df = df.drop(columns=['_kr_ratio'])

    non_empty = (df['company_web_text'].str.len() > 100).sum()
    print(f'enrichment 성공: {non_empty:,}/{len(target):,} (본문 100자 이상 확보)')
else:
    print('ENRICH_VIA_SEARCH=False — 웹 검색 enrichment skip')

ENRICH_VIA_SEARCH=False — 웹 검색 enrichment skip


In [39]:
# === [4단계] Gemini 구조화 추출 — 5관점 매칭 schema + hard prompt ===
#
# 100개 JD 샘플 분석 + user_data taxonomy 기반 설계:
# (1) user.categoryName = 자기소개서 카테고리 (인턴/프로젝트/대외활동/공모전 등) → jd_experience_requirements와 매칭 (B 0.5)
# (2) user.interestedJobs_1 = 18개 정형 직무 코드 → jd_job_role enum (A·B·C·D role 신호)
# (3) user.interestedIndustries_1 = 25 정형 산업 코드 → jd_industry enum (A·C·D industry)
# (4) user.ability_*_keyword = 12개 정형 역량 → jd_competencies enum (B·E)
# (5) detail_text_clean이 항상 채워짐 (100/100), duties_clean는 31%, skills/benefits는 0% → detail이 주신호
# (6) 섹션 헤더 패턴 활용: [모집부문] [담당업무] [주요업무] [자격요건] [우대사항] [필수자격]

USER_DATA_PATH = RAW_DATA / 'user_data.csv'
udf = pd.read_csv(USER_DATA_PATH, encoding='utf-8-sig', low_memory=False, usecols=[
    'categoryName',
    'interestedJobs_1', 'interestedJobs_2', 'interestedJobs_3',
    'interestedIndustries_1', 'interestedIndustries_2', 'interestedIndustries_3',
    'ability_0_keyword', 'ability_1_keyword', 'ability_2_keyword',
])

# user taxonomy를 빈도순 정렬 후 잡음 컷오프 (n>=10만)
def taxonomy_from(series_concat, min_count=10):
    vc = series_concat.dropna().astype(str).str.strip().value_counts()
    return sorted([v for v, n in vc.items() if n >= min_count and v not in ('nan', '', 'none')])

JOB_TAXONOMY = taxonomy_from(pd.concat([udf['interestedJobs_1'], udf['interestedJobs_2'], udf['interestedJobs_3']]), min_count=10)
INDUSTRY_TAXONOMY = taxonomy_from(pd.concat([udf['interestedIndustries_1'], udf['interestedIndustries_2'], udf['interestedIndustries_3']]), min_count=10)
COMPETENCY_TAXONOMY = taxonomy_from(pd.concat([udf['ability_0_keyword'], udf['ability_1_keyword'], udf['ability_2_keyword']]), min_count=50)
EXPERIENCE_TAXONOMY = taxonomy_from(udf['categoryName'], min_count=30)

print(f'JOB_TAXONOMY ({len(JOB_TAXONOMY)}): {JOB_TAXONOMY}')
print(f'\nINDUSTRY_TAXONOMY ({len(INDUSTRY_TAXONOMY)}): {INDUSTRY_TAXONOMY}')
print(f'\nCOMPETENCY_TAXONOMY ({len(COMPETENCY_TAXONOMY)}): {COMPETENCY_TAXONOMY}')
print(f'\nEXPERIENCE_TAXONOMY ({len(EXPERIENCE_TAXONOMY)}): {EXPERIENCE_TAXONOMY}')

# ============================================================
# JD_PROFILE_SCHEMA — 5관점 채점 가중치에 정밀 fit (19컬럼)
# ============================================================
JD_PROFILE_SCHEMA = {
    'type': 'object',
    'properties': {
        # === [A·B·C·D role 신호] user.interestedJobs_*와 동일 공간 ===
        'jd_job_role': {'type': 'string', 'enum': JOB_TAXONOMY + ['unknown'],
            'description': '주된 직무 코드. user interestedJobs_1과 동일 enum.'},
        'jd_job_role_secondary': {'type': 'array', 'items': {'type': 'string', 'enum': JOB_TAXONOMY},
            'description': '부수 직무 코드 max 3 (회사가 멀티 채용 시).'},
        'jd_role_specifics': {'type': 'array', 'items': {'type': 'string'},
            'description': '회사 내 구체 포지션명 max 8. 예: ["CAE 해석", "교육운영", "권상기 전기설계"]. [모집부문] 섹션에서 추출.'},

        # === [A·C·D industry 신호] user.interestedIndustries_*와 동일 ===
        'jd_industry': {'type': 'string', 'enum': INDUSTRY_TAXONOMY + ['unknown'],
            'description': '산업 코드. user interestedIndustries_1과 동일 enum. 회사명+사업 영역으로 판단.'},

        # === [A·B·C skill 신호] 실제 기술명 ===
        'jd_required_skills': {'type': 'array', 'items': {'type': 'string'},
            'description': '필수 기술/툴/자격증/언어명 max 15. 한국어 또는 영문 그대로 (Python, React, SQL, AWS, 포토샵, 운전면허). 일반 역량은 jd_competencies로.'},
        'jd_preferred_skills': {'type': 'array', 'items': {'type': 'string'},
            'description': '우대 스킬 max 10.'},

        # === [B 0.5 핵심] star 신호 ⭐ user.categoryName + ability_*_keyword와 매칭 ===
        'jd_experience_requirements': {'type': 'array', 'items': {'type': 'string', 'enum': EXPERIENCE_TAXONOMY},
            'description': '★ JD가 요구하는 경험 유형 max 4. user categoryName과 동일 enum. 예: ["인턴", "프로젝트", "대외활동"]. JD에 "~경험자" / "~경험 보유" / "~활동 경험" 류 표현에서 추출.'},
        'jd_competencies': {'type': 'array', 'items': {'type': 'string', 'enum': COMPETENCY_TAXONOMY},
            'description': 'user ability_*_keyword와 동일 공간 max 5. JD가 강조하는 역량.'},

        # === [D 0.3 context 신호] ⭐ NEW ===
        'jd_location': {'type': 'string',
            'description': '근무지 *큰 단위* (서울/경기/부산/판교/원격/전국). 건물명·층호는 제외. 모르면 unknown.'},
        'jd_company_size_tier': {'type': 'string', 'enum': ['large_corp', 'mid_corp', 'startup', 'public_org', 'foreign_corp', 'unknown'],
            'description': '회사 규모/유형. large_corp=대기업·계열사, mid_corp=중견, startup=초기·시리즈, public_org=공공·재단, foreign_corp=외국계. 회사명·톤·복지로 추론.'},
        'jd_work_format': {'type': 'string', 'enum': ['office', 'remote', 'hybrid', 'field', 'flexible', 'unknown'],
            'description': '근무 형태. field=현장/공장, flexible=자율출퇴근·탄력근무 명시. 단서 없으면 unknown.'},

        # === [임베딩 매칭용 의미 텍스트] B Resume-Centric의 star 신호 보강 ===
        'jd_company_intro': {'type': 'string',
            'description': '회사가 무엇 하는지 사실 1~3문장. 홍보 수식어(국내 최고/혁신 선도) X.'},
        'jd_role_purpose': {'type': 'string',
            'description': '이 직무가 *왜 필요한지* 1~2문장. 어떤 가치/문제 해결.'},
        'jd_main_duties_text': {'type': 'string',
            'description': '주요 업무 5~8문장 자연어. user STAR(Action)과 의미 비교 가능하게 *구체 활동·도구·범위* 포함. bullet 나열 X.'},
        'jd_ideal_candidate_text': {'type': 'string',
            'description': '자격·우대·인재상 3~6문장 통합. user 자기소개서 매칭의 직접 비교 대상.'},
        'jd_culture_keywords': {'type': 'array', 'items': {'type': 'string'},
            'description': '문화 시그널 max 6 (자율/수평적/데이터드리븐/빠른실행/글로벌/안정 등). 복리후생 명사 X.'},

        # === 메타 ===
        'jd_experience_level': {'type': 'string', 'enum': ['new', 'experienced', 'both', 'intern', 'contract', 'unknown']},
        'jd_posting_kind': {'type': 'string', 'enum': ['job_posting', 'training_program', 'internship_program', 'competition', 'company_promotion', 'low_quality', 'unknown']},
        'jd_summary': {'type': 'string', 'description': '한 줄 핵심 요약'},
        'jd_quality_flags': {'type': 'array', 'items': {'type': 'string'}, 'description': 'mojibake/short_text/low_signal/foreign_lang 등'},
    },
    'required': [
        'jd_job_role', 'jd_industry', 'jd_required_skills',
        'jd_experience_requirements', 'jd_competencies',
        'jd_location', 'jd_company_size_tier', 'jd_work_format',
        'jd_company_intro', 'jd_role_purpose', 'jd_main_duties_text', 'jd_ideal_candidate_text',
        'jd_experience_level', 'jd_posting_kind', 'jd_summary',
    ],
}


def build_jd_text(row) -> str:
    """Gemini 입력 — 정제된 텍스트 + 스크랩(있으면)."""
    parts = [
        f"회사명: {row.get('company_name', '')}",
        f"공고명: {row.get('title_v2', '')}",
        f"고용/경력 태그: {row.get('job_types', '')}",
        f"직무 정보(duties): {str(row.get('duties_clean_v2', ''))[:1500]}",
        f"자격/스킬: {str(row.get('skills_clean_v2', ''))[:800]}",
        f"상세 공고: {str(row.get('detail_text_clean_v2', ''))[:4000]}",
    ]
    scraped = str(row.get('scraped_text', ''))
    if scraped and len(scraped) > 50:
        parts.append(f"공고 내 URL 본문: {scraped[:2000]}")
    company_web = str(row.get('company_web_text', ''))
    if company_web and len(company_web) > 100:
        parts.append(f"회사 웹검색 보강 본문: {company_web[:2500]}")
    return '\n'.join(parts)


def jd_profile_prompt(jd_text: str) -> str:
    return f"""# 역할
너는 한국어 채용공고를 **5관점 매칭용**으로 전처리하는 모델이다.
최종 목적은 user의 자기소개서(자기소개서 본문 + STAR + 정형 역량 + 관심 직무·산업)와 이 채용공고를 *5가지 관점*에서 매칭하는 것이다.

# 5관점 매칭 가중치 (네가 추출하는 각 컬럼이 어느 관점에 쓰이는지)
- A Job-Centric (role 0.6 + skill 0.3 + industry 0.1) → jd_job_role, jd_role_specifics, jd_required_skills, jd_industry
- B Resume-Centric (star 0.5 + skill 0.3 + role 0.2) → **jd_experience_requirements**, jd_competencies, jd_main_duties_text, jd_ideal_candidate_text, jd_required_skills, jd_job_role
- C Skill-Centric (skill 0.7 + role 0.2 + industry 0.1) → **jd_required_skills (가장 정밀하게)**, jd_preferred_skills, jd_job_role, jd_industry
- D Context-Fit (industry 0.5 + role 0.2 + context 0.3) → jd_industry, **jd_location, jd_company_size_tier, jd_work_format, jd_culture_keywords**, jd_job_role
- E Mixed → 위 모두 균등

# JD 텍스트 섹션 헤더 (이 신호를 잘 활용해라)
다음 한국어 섹션 헤더가 보이면 의미 분리에 활용:
- [모집부문] / [채용분야] / [모집직무] → jd_role_specifics + jd_job_role
- [담당업무] / [주요업무] / [업무내용] → jd_main_duties_text
- [자격요건] / [필수자격] / [지원자격] → jd_ideal_candidate_text + jd_required_skills + jd_experience_requirements
- [우대사항] / [우대조건] → jd_ideal_candidate_text + jd_preferred_skills + jd_competencies
- [인재상] / [회사소개] / [브랜드 소개] → jd_company_intro + jd_culture_keywords
- [근무조건] / [근무지역] / [근무환경] → jd_location + jd_work_format
- [복리후생] / [혜택] → jd_culture_keywords (단 명사 나열 X, 문화 시그널만)
- [전형절차] / [지원기간] / [지원방법] / [유의사항] / [문의] → ❌ 모두 제외

# 절대 보존할 의미 (user 매칭의 핵심)
- 회사 정체성·사업·제품·미션
- 채용 직무의 본질·존재 이유
- 주요 업무 (구체 활동·도구·범위)
- 자격요건·우대·인재상 (자기소개서와 의미 비교 가능하게)
- 회사 문화 시그널 (자율/수평적/데이터드리븐/빠른실행 등)
- 사용 기술·스킬·자격증·언어
- **경험 요구사항** (인턴/프로젝트/대외활동/공모전/리더십 경험 등 — B 관점 0.5 핵심)

# 절대 제외할 행정/홍보 noise
- 이메일·전화·문의처
- 접수 기간/마감일/접수 방법 (2026-04-01 ~ 채용 시 마감, 간편지원)
- 전형 단계 (Step 1 서류전형 > 1차 면접 > 최종합격)
- 유의사항 boilerplate (허위 사실 발견 시 채용 취소)
- 모집 인원 ((0명), 00명 채용)
- 회사 주소 세부 (건물명·층호. 단 시/구 단위는 jd_location에)
- 복리후생 *명사 나열* (점심 1.3만원, 음료 간식 제공) — 단 자율출퇴근 등 *문화*는 jd_culture_keywords와 jd_work_format에
- 회사 홍보 수식어 (국내 최고의, 혁신을 선도하는)

# enum 강제 규칙
- jd_job_role / jd_industry / jd_competencies / jd_experience_requirements는 schema enum 중에서만 선택. 가까운 것이 없으면 빈 list 또는 'unknown'. 절대 새 값을 만들지 말 것.
- 회사 규모 추론 (jd_company_size_tier): 회사명 + 톤으로 판단.
  - large_corp: 삼성/현대/LG/SK/CJ/하이브/카카오/네이버 등 대기업·계열사
  - public_org: 한국XXX/공공/재단/공사
  - foreign_corp: 영문 회사명 + 한국지사·코리아
  - startup: 시리즈/투자 받은/창업한/스타트업 명시
  - mid_corp: 위에 해당하지 않는 일반 기업
  - unknown: 단서 없음

# 의미 보존 가이드 (5개 텍스트 컬럼)
- jd_company_intro: 회사가 *무엇을 하는지* 사실적으로 1~3문장.
- jd_role_purpose: 이 직무가 *왜 필요한지* 1~2문장.
- jd_main_duties_text: 주요 업무 *5~8문장 자연어 흐름*. 단순 bullet 나열 X. user STAR Action과 의미 비교 가능하게 *구체 활동·도구·범위* 포함.
- jd_ideal_candidate_text: 자격·우대·인재상 *3~6문장 자연어 통합*. user 자기소개서 비교 대상.
- jd_culture_keywords: 텍스트에 *명시적으로 드러난* 문화 시그널만 max 6.

# Few-shot 골든 예제 (신한은행 디지털/ICT 케이스)
입력 발췌: "신한은행 2021년 디지털/ICT 수시채용 [직무분야 및 필요역량] 신기술 활용 서비스 발굴 및 개발 -머신러닝/딥러닝 지식 및 활용 역량 -데이터 마이닝 -Python, Java, C/C++ -비즈니스 문제해결능력 -모바일 채널 서비스 개발 -iOS Platform -Swift, SwiftUI -React.js"
기대 출력 요약:
- jd_job_role: "software_dev"  (또는 "data_ai_ml" — 둘 다 강하면 jd_job_role_secondary에 보조)
- jd_industry: "finance_fintech"
- jd_required_skills: ["Python", "Java", "C/C++", "Swift", "SwiftUI", "React.js", "머신러닝", "딥러닝", "데이터 마이닝", "iOS", "HTML", "CSS", "JavaScript", "Git", "GitHub"]
- jd_competencies: ["문제해결 능력", "디지털·기술 활용 역량", "커뮤니케이션 역량"]
- jd_experience_requirements: ["프로젝트"] (개발/구현 경험 시그널)
- jd_company_size_tier: "large_corp"
- jd_work_format: "office"
- jd_location: "서울"
- jd_posting_kind: "job_posting"
- jd_company_intro: "신한은행은 국내 주요 시중은행 중 하나로 디지털/ICT 부문에서 신기술 활용 금융 서비스를 개발한다."
- jd_role_purpose: "머신러닝·딥러닝과 모바일 기술을 활용해 금융 고객 서비스를 발굴·개발하는 기술 직무."
- jd_main_duties_text: "신기술 활용 서비스 발굴 및 개발, 머신러닝/딥러닝 지식을 활용한 분석 모델 구축, 데이터 마이닝 기반 인사이트 도출, 컨테이너 기반 플랫폼 개발 및 운영, Python·Java·C/C++ 등 다양한 언어로 비즈니스 문제해결, 모바일 채널 서비스 개발 및 운영, iOS Platform과 Swift·SwiftUI 활용 앱 개발, React.js 등 모바일 웹 Framework 개발."
- jd_ideal_candidate_text: "머신러닝·딥러닝 지식과 데이터 분석 역량을 보유한 인재. iOS·Swift·React 등 모바일 개발 경험. Python·Java·C/C++ 등 다양한 프로그래밍 언어 활용 가능. Git/GitHub 협업 경험. 비즈니스 문제해결능력과 원활한 커뮤니케이션 역량."
- jd_culture_keywords: ["기술중심", "데이터드리븐"]

# Few-shot 골든 예제 2 (퓨어루 MD — 행정 noise 제거 케이스)
입력 발췌: "송지랩스는 무자극 영유아 스킨케어 브랜드 퓨어루를 운영. 자율 출퇴근 09~10시. 수평적 호칭. 데이터 분석 능력. AI 툴 활용. Step 1 서류전형 > 최종합격. 접수기간 2026-04-01 ~ 채용 시 마감. 문의 jhpark@sj-labs.kr"
기대 출력 요약:
- jd_job_role: "marketing" (보조: "sales")
- jd_industry: "beauty_cosmetics"
- jd_culture_keywords: ["자율", "수평적", "데이터드리븐", "스타트업"]
- jd_work_format: "flexible"
- jd_company_size_tier: "startup"
- jd_experience_requirements: ["인턴", "프로젝트"] (MD/마케팅 경험)
- ❌ 'Step 1', '접수기간 2026', 'jhpark@sj-labs' 같은 행정 정보는 *모든 컬럼에서 제외*

# 출력 형식
위 JSON schema에 맞는 JSON만 반환. 다른 텍스트·설명·코드블록 표시 일절 금지.

# JD 원문
{jd_text}""".strip()


# Prompt 길이 확인
_sample = jd_profile_prompt('회사명: 송지랩스\n공고명: [퓨어루] MD 주니어')
print(f'\n=== Prompt 길이: {len(_sample):,} chars (≈ {len(_sample)/3:.0f} tokens) ===')
print('첫 1500자 미리보기:')
print(_sample[:1500])


JOB_TAXONOMY (23): ['consulting', 'cs_cx', 'data_ai_ml', 'design', 'education', 'engineering_hw', 'finance_investment', 'hr', 'infrastructure_eng', 'logistics_scm', 'management_support', 'manufacturing', 'marketing', 'pharmaceutical_sales', 'planning_strategy', 'procurement', 'quality', 'rnd', 'robotics', 'sales', 'semiconductor', 'service_pm', 'software_dev']

INDUSTRY_TAXONOMY (40): ['aerospace_defense', 'ai_data', 'automotive', 'battery_energy', 'beauty_cosmetics', 'bio_healthcare', 'bio_pharma', 'chemical_material', 'chemical_materials', 'construction', 'construction_real_estate', 'consulting', 'consulting_professional', 'edu_tech', 'electronics', 'energy', 'fashion', 'finance_fintech', 'finance_insurance', 'fintech', 'fintech_finance', 'food_beverage', 'healthcare', 'it_software', 'logistics', 'logistics_transport', 'machinery_heavy', 'manufacturing', 'media_entertainment', 'medical', 'mobility', 'pharmaceutical', 'public_ngo', 'real_estate', 'retail_distribution', 'retail_ecommer

In [40]:
# === Gemini 호출 + 캐시 ===

_client = None
def get_client():
    global _client
    if _client is None and HAS_GEMINI and GEMINI_API_KEY:
        _client = genai.Client(api_key=GEMINI_API_KEY)
    return _client


def call_gemini_json(prompt: str, schema: dict, model: str = GEMINI_MODEL, max_retries: int = 3) -> dict:
    client = get_client()
    if client is None:
        raise RuntimeError('Gemini 클라이언트 없음 (HAS_GEMINI / API key 확인)')
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_json_schema=schema,
                    temperature=0.0,
                ),
            )
            return json.loads(resp.text)
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'Gemini 호출 실패: {last_err}')


def build_jd_profile_v2(row) -> dict:
    job_id = int(row['job_id'])
    cache_path = GEMINI_CACHE / f'{job_id}.json'
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding='utf-8'))

    if not HAS_GEMINI or not GEMINI_API_KEY:
        # API 미설정 — 빈 profile 반환 (캐시 안 함)
        return {k: 'unknown' if v.get('type') == 'string' else [] for k, v in JD_PROFILE_SCHEMA['properties'].items()}

    text = build_jd_text(row)
    profile = call_gemini_json(jd_profile_prompt(text), JD_PROFILE_SCHEMA)
    profile['_model'] = GEMINI_MODEL
    profile['_input_len'] = len(text)
    cache_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding='utf-8')
    return profile

In [41]:
# === Dry run: 상위 3행만 호출 ===
if HAS_GEMINI and GEMINI_API_KEY:
    for i in range(min(3, len(df))):
        row = df.iloc[i]
        print(f'\n--- job_id={row["job_id"]} | {row["company_name"]} | {row["title"]} ---')
        try:
            p = build_jd_profile_v2(row)
            for k, v in p.items():
                if k.startswith('_'): continue
                vs = str(v)[:120]
                print(f'  {k:>22s}: {vs}')
        except Exception as e:
            print(f'  ❌ ERROR: {e}')
else:
    print('Gemini 미설정 — dry-run skip')


--- job_id=312239 | 유한킴벌리(주) | [유한킴벌리] 2026년 유한킴벌리 신입 및 경력사원 채용 ---
             jd_job_role: marketing
   jd_job_role_secondary: ['rnd']
       jd_role_specifics: ['플랫폼 마케터', 'Research Scientist(육아용품 제품개발)']
             jd_industry: beauty_cosmetics
      jd_required_skills: ['데이터 분석', 'AI 툴']
     jd_preferred_skills: []
  jd_experience_requirements: ['인턴', '아르바이트', '프로젝트']
         jd_competencies: ['문제해결 능력', '디지털·기술 활용 역량', '실행력·업무처리 능력']
             jd_location: 서울
    jd_company_size_tier: large_corp
          jd_work_format: unknown
        jd_company_intro: 유한킴벌리는 생활용품 및 육아용품을 제조하고 판매하는 기업으로, 고객의 삶에 필요한 제품과 서비스를 제공합니다.
         jd_role_purpose: 이 채용은 유한킴벌리의 고객사업부문에서 플랫폼 마케팅을 통해 비즈니스 성장을 주도하고, 서초연구소에서 육아용품 제품 개발을 통해 혁신적인 제품을 선보이는 것을 목표로 합니다.
     jd_main_duties_text: 고객사업부문 플랫폼 마케터는 마케팅 전략 수립 및 기획, 온라인 마케팅 실행, 시장 조사 및 분석을 통해 브랜드 매니징을 수행합니다. 또한 광고 기획, 해외 마케팅, 언론 홍보 및 PR 활동을 전개하며, 설문 및 리
  jd_ideal_candidate_text: 지원자는 유한킴벌리에 대한 명확한 지원 동기와 단기 및 장기적인 입사 후 포부를 제시해야 합니다. 학교생활, 인턴

In [42]:
# === 본격 처리: 전체 113k 행 (또는 SAMPLE_SIZE 만큼) ===
# - 중간 체크포인트: 매 CHECKPOINT_EVERY 행마다 partial CSV 저장 (중단 대비)
# - 캐시 활용: 이미 호출된 행은 raw/data/gemini_cache/jd_profile_v2/{job_id}.json에서 즉시 로드
# - 중단·재실행: 같은 셀 다시 실행하면 캐시된 행은 모두 skip하고 남은 행만 호출

CHECKPOINT_EVERY = 1000
CHECKPOINT_PATH = RAW_DATA / 'company_jobdescription_enriched.partial.csv'

target_df = df.head(SAMPLE_SIZE) if SAMPLE_SIZE else df
n_target = len(target_df)
n_cached_pre = len(list(GEMINI_CACHE.glob('*.json')))
print(f'처리 대상  : {n_target:,} 행 (SAMPLE_SIZE={SAMPLE_SIZE})')
print(f'캐시 사전  : {n_cached_pre:,} 개 → 이만큼은 즉시 통과')
print(f'체크포인트 : 매 {CHECKPOINT_EVERY:,} 행마다 → {CHECKPOINT_PATH.name}')
print(f'예상 신규 호출: 최대 {max(0, n_target - n_cached_pre):,} (중복 캐시 hit 제외)')

profiles = []
failures = []
n_cached_hit = 0
n_api_called = 0
t0 = time.time()

for idx_pos, (_, row) in enumerate(tqdm(target_df.iterrows(), total=n_target, desc='gemini extract')):
    job_id = int(row['job_id'])
    cache_path = GEMINI_CACHE / f'{job_id}.json'
    was_cached = cache_path.exists()
    try:
        profiles.append(build_jd_profile_v2(row))
        if was_cached:
            n_cached_hit += 1
        else:
            n_api_called += 1
    except Exception as e:
        failures.append((job_id, str(e)[:200]))
        empty = {k: 'unknown' if v.get('type') == 'string' else [] for k, v in JD_PROFILE_SCHEMA['properties'].items()}
        profiles.append(empty)

    # 중간 체크포인트
    if (idx_pos + 1) % CHECKPOINT_EVERY == 0:
        try:
            partial_profile_df = pd.json_normalize(profiles)
            for col in partial_profile_df.columns:
                if partial_profile_df[col].apply(lambda x: isinstance(x, list)).any():
                    partial_profile_df[col] = partial_profile_df[col].apply(
                        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
                    )
            partial_out = pd.concat([target_df.head(idx_pos + 1).reset_index(drop=True),
                                     partial_profile_df.reset_index(drop=True)], axis=1)
            partial_out.to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')
            elapsed = time.time() - t0
            rate = (idx_pos + 1) / elapsed if elapsed > 0 else 0
            eta_sec = (n_target - idx_pos - 1) / rate if rate > 0 else 0
            tqdm.write(
                f'  [ckpt @ {idx_pos+1:,}/{n_target:,}] '
                f'cache_hit={n_cached_hit:,} api_called={n_api_called:,} failed={len(failures):,} '
                f'| {rate:.1f} rows/s | ETA {eta_sec/3600:.1f}h'
            )
        except Exception as e:
            tqdm.write(f'  ⚠️ checkpoint 저장 실패: {e}')

elapsed_total = time.time() - t0
print(f'\n완료: {len(profiles):,} 행')
print(f'  - 캐시 hit  : {n_cached_hit:,}')
print(f'  - API 호출  : {n_api_called:,}')
print(f'  - 실패      : {len(failures):,}')
print(f'  - 총 소요   : {elapsed_total/60:.1f} 분 ({len(profiles)/max(1,elapsed_total):.1f} rows/s)')
if failures[:3]:
    print(f'  처음 3개 실패: {failures[:3]}')

처리 대상  : 84,930 행 (SAMPLE_SIZE=None)
캐시 사전  : 10 개 → 이만큼은 즉시 통과
체크포인트 : 매 1,000 행마다 → company_jobdescription_enriched.partial.csv
예상 신규 호출: 최대 84,920 (중복 캐시 hit 제외)


gemini extract:   1%|          | 1000/84930 [4:04:27<334:30:41, 14.35s/it]

  [ckpt @ 1,000/84,930] cache_hit=10 api_called=990 failed=0 | 0.1 rows/s | ETA 342.0h


gemini extract:   2%|▏         | 2000/84930 [7:48:05<245:38:27, 10.66s/it] 

  [ckpt @ 2,000/84,930] cache_hit=10 api_called=1,710 failed=280 | 0.1 rows/s | ETA 323.5h


gemini extract:   4%|▎         | 3000/84930 [10:46:57<242:00:24, 10.63s/it]

  [ckpt @ 3,000/84,930] cache_hit=10 api_called=1,711 failed=1,279 | 0.1 rows/s | ETA 294.5h


gemini extract:   4%|▍         | 3388/84930 [12:56:57<311:39:53, 13.76s/it]


KeyboardInterrupt: 

In [43]:
# === [5단계] 최종 정리 + 단일 깨끗한 CSV 저장 ===
# 정리 규칙:
#   1) 무용지물 컬럼 삭제 (100% NaN인 skills_clean / benefits_clean / scraped/company_web_text [옵션이 False였을 때])
#   2) 중복 컬럼 통합 (title ↔ title_v2 → 정제본만, 원본 컬럼 삭제)
#   3) 빈 값 일관: string='unknown' 또는 '' / list='[]' / NaN 제거
#   4) ★ jd_matching_text 신규 — 의미 컬럼 4개 합본 (임베딩 입력용)
#   5) 컬럼 순서 = 5관점 매핑 순서로 정렬

# 1. profile DataFrame 구성
profile_df = pd.json_normalize(profiles)
for col in profile_df.columns:
    if profile_df[col].apply(lambda x: isinstance(x, list)).any():
        profile_df[col] = profile_df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
        )

out_df = pd.concat([target_df.reset_index(drop=True), profile_df.reset_index(drop=True)], axis=1)
out_df['extracted_urls'] = out_df['extracted_urls'].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)

# 2. 무용지물·중복 컬럼 삭제 ---------------------------------------------
# 채움률 0%인 컬럼 동적 탐지
def is_dead_col(s):
    rate = ((s.notna()) & (s.astype(str).str.strip() != '') &
            (~s.astype(str).str.lower().isin(['nan','null','none','unknown','[]','[ ]']))).mean()
    return rate < 0.005   # 0.5% 미만이면 dead

DEAD_CANDIDATES = ['skills_clean', 'skills_clean_v2', 'benefits_clean', 'benefits_clean_v2',
                   'scraped_text', 'company_web_text']
drop_cols = [c for c in DEAD_CANDIDATES if c in out_df.columns and is_dead_col(out_df[c])]

# 항상 제거: 중복 원본 + 메타
ALWAYS_DROP = ['title', 'duties_clean', 'detail_text_clean', '_model', '_input_len', 'has_url']
drop_cols.extend([c for c in ALWAYS_DROP if c in out_df.columns])

out_df = out_df.drop(columns=list(set(drop_cols)))
print(f'삭제된 컬럼: {sorted(set(drop_cols))}')

# 3. 정제본 컬럼 rename ---------------------------------------------
out_df = out_df.rename(columns={
    'title_v2': 'title',
    'duties_clean_v2': 'duties_raw',
    'detail_text_clean_v2': 'detail_text',
})

# 4. jd_matching_text — 의미 컬럼 4개 합본 (임베딩 입력용) ----------
def build_matching_text(row):
    parts = []
    for col, label in [
        ('jd_company_intro', '회사'),
        ('jd_role_purpose', '직무 목적'),
        ('jd_main_duties_text', '주요 업무'),
        ('jd_ideal_candidate_text', '인재상'),
    ]:
        v = row.get(col, '')
        s = str(v).strip() if v is not None else ''
        if s and s.lower() not in ('', 'unknown', 'nan', 'none', 'null'):
            parts.append(f'{label}: {s}')
    return '\n'.join(parts)

out_df['jd_matching_text'] = out_df.apply(build_matching_text, axis=1)

# 5. 빈 값 일관 처리 ---------------------------------------------
LIST_COLS = [
    'jd_job_role_secondary', 'jd_role_specifics', 'jd_required_skills',
    'jd_preferred_skills', 'jd_experience_requirements', 'jd_competencies',
    'jd_culture_keywords', 'jd_quality_flags', 'extracted_urls',
]
for c in LIST_COLS:
    if c in out_df.columns:
        out_df[c] = (out_df[c].fillna('[]').astype(str)
                     .replace({'': '[]', 'nan': '[]', 'None': '[]', 'null': '[]'}))

ENUM_STRING_COLS = [
    'jd_job_role', 'jd_industry', 'jd_location',
    'jd_company_size_tier', 'jd_work_format',
    'jd_experience_level', 'jd_posting_kind',
]
for c in ENUM_STRING_COLS:
    if c in out_df.columns:
        out_df[c] = (out_df[c].fillna('unknown').astype(str)
                     .replace({'': 'unknown', 'nan': 'unknown', 'None': 'unknown', 'null': 'unknown'}))

FREE_TEXT_COLS = [
    'jd_company_intro', 'jd_role_purpose', 'jd_main_duties_text',
    'jd_ideal_candidate_text', 'jd_summary', 'duties_raw', 'detail_text', 'jd_matching_text',
]
for c in FREE_TEXT_COLS:
    if c in out_df.columns:
        out_df[c] = (out_df[c].fillna('').astype(str)
                     .replace({'nan': '', 'None': '', 'null': '', 'unknown': ''}))

# 6. 컬럼 순서 = 5관점 매핑 순서 ---------------------------------------------
CLEAN_ORDER = [
    # === 식별자 ===
    'job_id', 'company_name', 'title', 'job_types',
    # === 원문 (정제본) ===
    'detail_text', 'duties_raw', 'extracted_urls',
    # === [A·B·C·D] role 신호 ===
    'jd_job_role', 'jd_job_role_secondary', 'jd_role_specifics',
    # === [A·C] skill 신호 ===
    'jd_required_skills', 'jd_preferred_skills',
    # === [A·C·D] industry + [D] context 신호 ===
    'jd_industry', 'jd_location', 'jd_company_size_tier', 'jd_work_format',
    # === [B] star 신호 (Resume-Centric 핵심) ===
    'jd_experience_requirements', 'jd_competencies',
    # === 의미 보존 텍스트 ===
    'jd_company_intro', 'jd_role_purpose', 'jd_main_duties_text', 'jd_ideal_candidate_text', 'jd_culture_keywords',
    # === ★ 임베딩 통합 입력 ===
    'jd_matching_text',
    # === 메타 ===
    'jd_experience_level', 'jd_posting_kind', 'jd_summary', 'jd_quality_flags',
]
present_cols = [c for c in CLEAN_ORDER if c in out_df.columns]
missing_cols = [c for c in out_df.columns if c not in present_cols]
if missing_cols:
    print(f'⚠️ CLEAN_ORDER에 없는 컬럼 (뒤에 붙임): {missing_cols}')
    present_cols += missing_cols
out_df = out_df[present_cols]

# 7. 저장 ---------------------------------------------
out_df.to_csv(DST, index=False, encoding='utf-8-sig')
print(f'\n✅ saved: {DST}')
print(f'   shape: {out_df.shape}')
print(f'   파일 크기: {DST.stat().st_size / (1024*1024):.1f} MB')

# 8. 채움률 통계 ---------------------------------------------
def fill_rate(s):
    return ((s.notna()) & (s.astype(str).str.strip() != '') &
            (~s.astype(str).str.lower().isin(['nan','null','none','unknown','[]','[ ]']))).mean() * 100

print('\n=== 컬럼별 채움률 (clean) ===')
for c in out_df.columns:
    r = fill_rate(out_df[c])
    bar = '█' * int(r/5) + '░' * (20 - int(r/5))
    print(f'  {bar} {r:5.1f}%  {c}')

print(f'\n=== 분포 ===')
for c in ['jd_job_role', 'jd_industry', 'jd_posting_kind', 'jd_company_size_tier']:
    if c in out_df.columns:
        print(f'\n[{c}]')
        print(out_df[c].value_counts().to_string())

삭제된 컬럼: ['_input_len', '_model', 'benefits_clean', 'benefits_clean_v2', 'company_web_text', 'detail_text_clean', 'duties_clean', 'has_url', 'scraped_text', 'skills_clean', 'skills_clean_v2', 'title']

✅ saved: raw/data/company_jobdescription_enriched.csv
   shape: (84930, 28)
   파일 크기: 113.8 MB

=== 컬럼별 채움률 (clean) ===
  ████████████████████ 100.0%  job_id
  ████████████████████ 100.0%  company_name
  ████████████████████ 100.0%  title
  ████████████████████ 100.0%  job_types
  ████████████████████ 100.0%  detail_text
  ███████░░░░░░░░░░░░░  36.3%  duties_raw
  █░░░░░░░░░░░░░░░░░░░   7.2%  extracted_urls
  ░░░░░░░░░░░░░░░░░░░░   2.0%  jd_job_role
  ░░░░░░░░░░░░░░░░░░░░   1.1%  jd_job_role_secondary
  ░░░░░░░░░░░░░░░░░░░░   2.0%  jd_role_specifics
  ░░░░░░░░░░░░░░░░░░░░   1.2%  jd_required_skills
  ░░░░░░░░░░░░░░░░░░░░   1.0%  jd_preferred_skills
  ░░░░░░░░░░░░░░░░░░░░   2.0%  jd_industry
  ░░░░░░░░░░░░░░░░░░░░   1.0%  jd_location
  ░░░░░░░░░░░░░░░░░░░░   1.9%  jd_company_size_tier
  ░░